# Capítulo 9 — Aplicações: inundação em NDN e evasão de antifraude

## Missão
Dois casos da parte IV do livro:
1. **Inundação de interesses (IFA)** em redes NDN: o atacante manda pedidos de conteúdo que não existe e enche as tabelas dos roteadores. Você vai detectar com características de janela e um classificador, avaliando em roteadores que o modelo nunca viu.
2. **Ataque ao antifraude**: o cap. 17 usa uma GAN para gerar fraudes que passam despercebidas. Aqui, você faz o *red team* do próprio modelo: uma busca de caixa preta que ajusta fraudes até elas passarem, e depois a defesa (retreinar com as fraudes evasivas e travar as características que o fraudador controla).

**No fim deste caderno você terá:** o módulo `meu_escudo/aplicacoes.py` (características de janela para NDN, a busca de evasão de caixa preta com características travadas e o retreino defensivo).

**Leitura essencial (opcional, livro em inglês):**
- 16.2 Preliminaries (NDN, IFA) · PDF p. 274
- 16.3 Machine Learning Assisted For NDN-Based IFA Detection In IOTSE · PDF p. 276
- 16.4 Performance Evaluation · PDF p. 280
- 17.2 Experiment Description · PDF p. 292
- 17.3 Generator And Discrimination Model · PDF p. 294

**Como usar:** cada passo tem uma explicação curta e uma célula vazia. Digite o código e rode com `Shift+Enter`. Travou por mais de 10 minutos? O mesmo passo pronto está em `../resolvidos/cap09_aplicacoes_inundacao_em_ndn_e_evasao_de_antif.ipynb`.

## Preparação
Configuração do caderno (já vem pronta). Rode antes de tudo.

In [ ]:
import os, sys, json, time, math, random
sys.path.insert(0, "../comum")                 # dados.py (dados fictícios do projeto)
sys.path.insert(0, "..")                       # raiz: onde fica o seu pacote meu_escudo/
os.makedirs("../meu_escudo", exist_ok=True)
open("../meu_escudo/__init__.py", "a").close()        # meu_escudo/ é o projeto que você constrói ao longo do livro
os.makedirs("../saidas", exist_ok=True)        # arquivos gerados nos exercícios
from dados import *
import warnings; warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2
print("Pronto. Python", sys.version.split()[0], "| seu pacote: meu_escudo")

## Ideia em 30 segundos
**IFA:** o sinal está na **taxa de interesses** e na fração de interesses pendentes não atendidos, mas o valor instantâneo é ruidoso. Médias móveis e desvio em relação à linha de base do próprio roteador ajudam.

**Evasão:**
- o fraudador mexe no que controla (valor, hora, número de transações);
- não mexe no que não controla (aparelho novo, distância do endereço);
- e consulta o modelo até a probabilidade de fraude cair abaixo do limiar.

A defesa é **retreinar com esses exemplos** e dar mais peso às características que o fraudador não consegue mudar.

### 9.1 **As aplicações.** Crie `aplicacoes.py` com:
- `caracteristicas_ndn(df, janela=5)`: por roteador (em ordem de `t`), a média móvel e o desvio móvel da taxa e do PIT, e a razão entre a taxa e a mediana dos 50 primeiros passos do roteador (a linha de base);
- `evadir(prever_proba, x, controlaveis, limites, limiar, passos=300, semente=0)`: uma busca aleatória que altera só as colunas controláveis, dentro dos limites, e aceita a mudança quando a probabilidade de fraude cai. Devolve `(x_novo, prob_final, consultas)`.

_A célula grava `meu_escudo/aplicacoes.py`: a 1ª linha (`%%writefile`) já vem pronta, digite o código embaixo dela._

In [ ]:
%%writefile ../meu_escudo/aplicacoes.py


### 9.2 **IFA: características e avaliação honesta.** Monte as características de `interesses_ndn()`. Separe os roteadores (não as linhas!) em treino (0 a 13) e teste (14 a 19) e treine uma `RandomForestClassifier(200)`. Compare com o uso só da taxa instantânea: precisão, revocação e F1 no teste.

### 9.3 **O antifraude.** Treine uma `RandomForestClassifier(300, min_samples_leaf=3)` em `transacoes_banco()` (70/30 estratificado). Escolha o limiar que dá 90% de revocação no teste e mostre a precisão nesse ponto.

### 9.4 **Red team: as fraudes evasivas.** Para 60 fraudes do teste que o modelo pega, rode o `evadir`, mexendo só em `valor` (de 50 a 3.000), `hora` (de 0 a 23) e `transacoes_24h` (de 0 a 10). Quantas passam? Com quantas consultas em média? Mostre como uma fraude típica foi alterada.

### 9.5 **A defesa.** Duas medidas:
- (a) **retreinar** juntando as fraudes evasivas (com rótulo 1) ao treino;
- (b) um modelo que dá mais peso ao que o fraudador **não controla**: treine só com `dispositivo_novo`, `distancia_km` e `tentativas_senha`.

Rode o mesmo ataque de evasão em 40 fraudes **novas** contra cada modelo e compare quantas passam, mantendo a revocação em 90% nas fraudes normais.

## Fechamento

- O que eu construí hoje:
- O que me surpreendeu:
- O que quero tentar sozinho a seguir:
